# Issue Salience and Partisan Electoral Outcomes
## Using BERTopic, VADER, and Zero-Shot Stance Classification

**Research Question:** To what extent do state-level issue salience profiles — defined by the dominant policy concerns driving voter behavior — predict partisan electoral outcomes across U.S. states in presidential elections?

**Theoretical Framework:** Issue Ownership Theory (Petrocik, 1996)

**Methods:**
1. **BERTopic** — Unsupervised topic modeling to identify policy issues in political tweets
2. **VADER** — Lexicon-based sentiment analysis for directional issue salience
3. **Zero-Shot Classification** — Transformer-based partisan stance detection
4. **Predictive Modeling** — Logistic Regression, Random Forest, Gradient Boosting
5. **Systematic Bias Audit** — Geographic, engagement, temporal, and fairness analysis

In [2]:
# Install dependencies (Colab-compatible)
!pip install -q datasets bertopic sentence-transformers vaderSentiment transformers torch scikit-learn plotly gensim umap-learn hdbscan tqdm kaleido huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.32.0 requires protobuf<5,>=3.20, but you have protobuf 5.29.4 which is incompatible.
streamlit 1.32.0 requires rich<14,>=10.14.0, but you have rich 15.0.0 which is incompatible.


In [ ]:
import pandas as pd
import numpy as np
import re
import json
import warnings
from tqdm import tqdm

import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook"
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import LeaveOneOut, cross_val_predict, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
SAMPLE_SIZE = 75000
np.random.seed(RANDOM_STATE)

# Detect GPU
import torch
DEVICE = 0 if torch.cuda.is_available() else -1
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Device for transformers: {DEVICE}")

c:\Users\semjo\anaconda3\Lib\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.




OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\semjo\anaconda3\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

## 1. Data Loading and Preprocessing

Data is loaded from HuggingFace: `Diogo2110/sem4data`. The dataset contains ~2.1M tweets collected during the 2024 U.S. election cycle, with metadata including user engagement metrics, state-level geolocation, and timestamps.

In [ ]:
from huggingface_hub import hf_hub_download

# Download the raw CSV directly (avoids Arrow schema parsing errors)
csv_path = hf_hub_download(
    repo_id="Diogo2110/sem4data",
    repo_type="dataset",
    filename="subjects_AND_sampling_metadata_anonymized_full.csv"
)

df = pd.read_csv(csv_path, low_memory=False)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
df.head(3)

In [ ]:
df = df.rename(columns={"clean...state_simple": "state"})

tweets_col = 'tweets_historical'

print(f"Total rows: {len(df)}")
print(f"Missing tweets: {df[tweets_col].isna().sum()}")
print(f"Missing states: {df['state'].isna().sum()}")
print(f"\nState distribution (top 10):")
print(df['state'].value_counts().head(10))

### Text Filtering and Cleaning

In [ ]:
df = df.dropna(subset=[tweets_col])
df = df[df[tweets_col].str.strip() != '']
df = df[~df[tweets_col].str.startswith("RT @")]
df = df[df[tweets_col].str.len() >= 20]
print(f"After filtering: {len(df)} tweets")

In [ ]:
def clean_tweet(text):
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["tweet_clean"] = df[tweets_col].apply(clean_tweet)
df = df[df["tweet_clean"].str.len() >= 15]
print(f"After cleaning: {len(df)} tweets")

### Stratified Sampling

We use a political keyword pre-filter to ensure our sample has high political signal density. Half the sample comes from tweets containing political keywords, and half from general tweets. This reduces the noise cluster problem from the baseline model.

In [ ]:
political_keywords = [
    'trump', 'biden', 'harris', 'kamala', 'democrat', 'republican', 'gop',
    'election', 'vote', 'voting', 'ballot', 'congress', 'senate', 'house',
    'policy', 'tax', 'taxes', 'immigration', 'immigrant', 'border',
    'abortion', 'gun', 'guns', 'climate', 'healthcare', 'health care',
    'economy', 'inflation', 'prices', 'jobs',
    'ukraine', 'israel', 'gaza', 'hamas', 'war',
    'maga', 'liberal', 'conservative', 'progressive',
    'supreme court', 'amendment', 'legislation', 'governor', 'president',
    'debate', 'campaign', 'rally', 'poll', 'polls',
    'democrat', 'republican', 'dnc', 'rnc', 'potus',
    'ice', 'deportation', 'welfare', 'social security', 'medicare'
]

pattern = '|'.join(political_keywords)
df['is_political'] = df['tweet_clean'].str.lower().str.contains(pattern, na=False)

n_political = df['is_political'].sum()
n_general = (~df['is_political']).sum()
print(f"Political tweets: {n_political:,}")
print(f"General tweets: {n_general:,}")

half = SAMPLE_SIZE // 2
df_pol = df[df['is_political']].sample(n=min(half, n_political), random_state=RANDOM_STATE)
df_gen = df[~df['is_political']].sample(n=min(half, n_general), random_state=RANDOM_STATE)
df_sample = pd.concat([df_pol, df_gen]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

docs = df_sample["tweet_clean"].tolist()
print(f"\nFinal sample size: {len(df_sample)}")
print(f"  Political: {df_sample['is_political'].sum()}")
print(f"  General: {(~df_sample['is_political']).sum()}")

## 2. BERTopic Topic Modeling

### Improvements over baseline:
- **HDBSCAN `min_cluster_size`**: reduced from 50 to 20 to capture smaller coherent clusters
- **HDBSCAN `min_samples`**: set to 10 for better outlier boundary detection
- **UMAP `n_neighbors`**: increased from 15 to 20 for richer local structure
- **Post-hoc outlier reduction**: using BERTopic's `reduce_outliers()` method
- **Stratified sampling**: political keyword pre-filtering increases signal density

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

umap_model = UMAP(
    n_neighbors=20,
    n_components=5,
    metric='cosine',
    random_state=RANDOM_STATE
)

hdbscan_model = HDBSCAN(
    min_cluster_size=20,
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    min_df=3
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=True,
    top_n_words=10,
    verbose=True
)

In [ ]:
topics, probs = topic_model.fit_transform(docs)

topic_info = topic_model.get_topic_info()
n_outliers_before = sum(1 for t in topics if t == -1)
pct_before = n_outliers_before / len(topics) * 100
print(f"Topics discovered: {len(topic_info) - 1}")
print(f"Outliers BEFORE reduction: {n_outliers_before} ({pct_before:.1f}%)")
print()
print(topic_info.head(15))

### Outlier Reduction

BERTopic's `reduce_outliers()` reassigns noise-cluster documents to their nearest topic based on c-TF-IDF similarity.

In [ ]:
new_topics = topic_model.reduce_outliers(docs, topics, strategy="c-tf-idf", threshold=0.1)
topic_model.update_topics(docs, topics=new_topics)

n_outliers_after = sum(1 for t in new_topics if t == -1)
pct_after = n_outliers_after / len(new_topics) * 100

print(f"Outliers BEFORE reduction: {n_outliers_before} ({pct_before:.1f}%)")
print(f"Outliers AFTER reduction:  {n_outliers_after} ({pct_after:.1f}%)")
print(f"Reduction: {n_outliers_before - n_outliers_after} tweets reassigned")

df_sample['topic'] = new_topics

topic_info_updated = topic_model.get_topic_info()
print(f"\nUpdated topic info:")
print(topic_info_updated.head(15))

### Model Evaluation

In [ ]:
cleaned_docs = [doc.split() for doc in docs]
dictionary = Dictionary(cleaned_docs)

topics_words = []
for topic_id in topic_model.get_topic_info()["Topic"]:
    if topic_id != -1:
        words = [word for word, _ in topic_model.get_topic(topic_id)]
        words = [word for word in words if word in dictionary.token2id]
        if len(words) > 0:
            topics_words.append(words)

coherence_model = CoherenceModel(
    topics=topics_words,
    texts=cleaned_docs,
    dictionary=dictionary,
    coherence="c_v"
)
coherence_score = coherence_model.get_coherence()

def topic_diversity(tw, top_n=10):
    unique_words = set()
    total_words = []
    for topic in tw:
        top_words = topic[:top_n]
        unique_words.update(top_words)
        total_words.extend(top_words)
    return len(unique_words) / len(total_words) if total_words else 0

diversity_score = topic_diversity(topics_words)

print("=" * 50)
print("MODEL EVALUATION")
print("=" * 50)
print(f"{'Metric':<25} {'Baseline':>10} {'Improved':>10}")
print("-" * 50)
print(f"{'Coherence (c_v)':<25} {'0.4609':>10} {coherence_score:>10.4f}")
print(f"{'Topic Diversity':<25} {'0.8972':>10} {diversity_score:>10.4f}")
print(f"{'Outlier %':<25} {'51.4%':>10} {pct_after:>9.1f}%")
print(f"{'Num Topics':<25} {'84':>10} {len(topics_words):>10}")
print("=" * 50)

### Topic Labeling and Classification

We inspect the discovered topics and assign human-readable labels. Topics are classified as political or non-political for downstream analysis.

In [ ]:
print("Top 20 topics by size:")
print("=" * 80)
for _, row in topic_info_updated.head(21).iterrows():
    topic_id = row['Topic']
    count = row['Count']
    if topic_id == -1:
        label = "NOISE"
    else:
        words = [w for w, _ in topic_model.get_topic(topic_id)][:5]
        label = ", ".join(words)
    print(f"  Topic {topic_id:>3}: {count:>5} tweets | {label}")

In [ ]:
# Map topics to political issue labels based on top words
# This mapping will need to be adjusted based on the actual discovered topics
topic_labels = {}
political_topic_ids = []

for topic_id in topic_model.get_topic_info()["Topic"]:
    if topic_id == -1:
        topic_labels[topic_id] = "Noise"
        continue
    words = [w for w, _ in topic_model.get_topic(topic_id)][:10]
    words_str = " ".join(words).lower()

    label = None
    is_political = False

    if any(w in words_str for w in ['israel', 'gaza', 'hamas', 'palestinian', 'jewish']):
        label = "Middle East / Israel-Gaza"
        is_political = True
    elif any(w in words_str for w in ['tax', 'inflation', 'economy', 'prices', 'economic']):
        label = "Economic Policy"
        is_political = True
    elif any(w in words_str for w in ['healthcare', 'health', 'patient', 'medical', 'insurance']):
        label = "Healthcare"
        is_political = True
    elif any(w in words_str for w in ['immigration', 'border', 'immigrant', 'ice', 'deport', 'migrant']):
        label = "Immigration / Border"
        is_political = True
    elif any(w in words_str for w in ['kamala', 'harris', 'trump', 'biden', 'election', 'vote', 'ballot']):
        label = "2024 Election / Candidates"
        is_political = True
    elif any(w in words_str for w in ['gun', 'shooting', 'firearm', 'amendment']):
        label = "Gun Policy"
        is_political = True
    elif any(w in words_str for w in ['climate', 'environment', 'energy', 'fossil', 'carbon']):
        label = "Climate / Environment"
        is_political = True
    elif any(w in words_str for w in ['abortion', 'roe', 'reproductive', 'pro life', 'pro choice']):
        label = "Abortion / Reproductive Rights"
        is_political = True
    elif any(w in words_str for w in ['ukraine', 'russia', 'nato', 'foreign policy', 'china']):
        label = "Foreign Policy"
        is_political = True
    elif any(w in words_str for w in ['police', 'crime', 'justice', 'prison', 'law enforcement']):
        label = "Criminal Justice"
        is_political = True
    elif any(w in words_str for w in ['education', 'school', 'student', 'college', 'university']):
        label = "Education"
        is_political = True
    elif any(w in words_str for w in ['democrat', 'republican', 'gop', 'liberal', 'conservative', 'maga']):
        label = "Partisan Discourse"
        is_political = True
    elif any(w in words_str for w in ['god', 'jesus', 'church', 'faith', 'christian', 'religion']):
        label = "Religion / Faith"
        is_political = False
    elif any(w in words_str for w in ['morning', 'good morning', 'good night', 'blessed']):
        label = "General Greetings"
        is_political = False
    elif any(w in words_str for w in ['game', 'team', 'player', 'season', 'football', 'nfl', 'nba']):
        label = "Sports"
        is_political = False
    elif any(w in words_str for w in ['thank', 'follow', 'giveaway', 'retweet']):
        label = "Spam / Engagement Bait"
        is_political = False
    elif any(w in words_str for w in ['taylor', 'swift', 'concert', 'album', 'music']):
        label = "Pop Culture"
        is_political = False
    else:
        label = f"Other ({', '.join(words[:3])})"
        is_political = False

    topic_labels[topic_id] = label
    if is_political:
        political_topic_ids.append(topic_id)

df_sample['topic_label'] = df_sample['topic'].map(topic_labels)

print(f"Political topics identified: {len(political_topic_ids)}")
print(f"\nPolitical topics:")
for tid in political_topic_ids:
    count = (df_sample['topic'] == tid).sum()
    print(f"  {tid:>3}: {topic_labels[tid]:<35} ({count} tweets)")

df_political = df_sample[df_sample['topic'].isin(political_topic_ids)].copy()
print(f"\nTotal political tweets: {len(df_political)} ({len(df_political)/len(df_sample)*100:.1f}%)")

### State-Level Topic Distributions

In [ ]:
state_name_to_code = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
    "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
    "Wisconsin": "WI", "Wyoming": "WY", "District of Columbia": "DC"
}

df_sample["state_code"] = df_sample["state"].map(state_name_to_code)

exclude_states = ["USA", "US", "United States", None, "Unknown"]
df_states = df_sample[~df_sample["state"].isin(exclude_states)].copy()
df_states = df_states.dropna(subset=["state_code"])

print(f"Tweets with valid state: {len(df_states)}")
print(f"States represented: {df_states['state_code'].nunique()}")

In [ ]:
df_pol_states = df_states[df_states['topic'].isin(political_topic_ids)].copy()

topic_state_counts = (
    df_pol_states.groupby(["topic_label", "state_code"])
    .size()
    .reset_index(name="count")
)

topic_state_counts["pct"] = (
    topic_state_counts.groupby("topic_label")["count"]
    .transform(lambda x: x / x.sum() * 100)
    .round(1)
)

fig = px.treemap(
    topic_state_counts,
    path=["topic_label", "state_code"],
    values="count",
    custom_data=["pct", "state_code"],
    title="States Grouped by Policy Issue",
    color="topic_label",
    color_discrete_sequence=px.colors.qualitative.Safe
)
fig.update_traces(
    texttemplate="%{customdata[1]}<br>%{customdata[0]}%",
    textposition="middle center"
)
fig.update_layout(height=700)
fig.show(renderer="notebook")

In [ ]:
state_topic = (
    df_pol_states.groupby(["state_code", "topic_label"])
    .size()
    .reset_index(name="count")
)
state_totals = df_pol_states.groupby("state_code").size().reset_index(name="total")
state_topic = state_topic.merge(state_totals, on="state_code")
state_topic["pct"] = (state_topic["count"] / state_topic["total"] * 100).round(1)

fig = px.bar(
    state_topic,
    x="state_code", y="pct", color="topic_label",
    title="Policy Issue Composition by State",
    labels={"pct": "% of tweets", "state_code": "State", "topic_label": "Policy Issue"},
    color_discrete_sequence=px.colors.qualitative.Safe
)
fig.update_layout(
    barmode="stack",
    xaxis=dict(tickangle=45, tickfont=dict(size=9)),
    yaxis=dict(title="% of tweets"),
    height=600,
    legend=dict(title="Policy Issue", orientation="v", yanchor="top", y=1, xanchor="left", x=1.01),
    margin=dict(t=80, b=120, l=60, r=200)
)
fig.show(renderer="notebook")

## 3. VADER Sentiment Analysis

VADER (Valence Aware Dictionary and sEntiment Reasoner) is well-suited for social media text. It handles slang, capitalization emphasis, and emoticons without requiring labeled training data. The compound score ranges from -1 (most negative) to +1 (most positive).

In [ ]:
analyzer = SentimentIntensityAnalyzer()

df_sample['vader_compound'] = df_sample['tweet_clean'].apply(
    lambda x: analyzer.polarity_scores(str(x))['compound']
)
df_sample['vader_label'] = df_sample['vader_compound'].apply(
    lambda x: 'positive' if x >= 0.05 else ('negative' if x <= -0.05 else 'neutral')
)

print("Overall sentiment distribution:")
print(df_sample['vader_label'].value_counts())
print(f"\nMean compound score: {df_sample['vader_compound'].mean():.4f}")
print(f"Median compound score: {df_sample['vader_compound'].median():.4f}")

### Sentiment by Political Topic

In [ ]:
df_pol_sent = df_sample[df_sample['topic'].isin(political_topic_ids)].copy()

topic_sentiment = df_pol_sent.groupby('topic_label')['vader_compound'].agg(['mean', 'median', 'std', 'count']).round(4)
topic_sentiment = topic_sentiment.sort_values('mean')
print(topic_sentiment)

fig = px.bar(
    topic_sentiment.reset_index(),
    x='topic_label', y='mean',
    error_y='std',
    title="Mean VADER Sentiment by Political Topic",
    labels={"mean": "Mean Compound Score", "topic_label": "Political Topic"},
    color='mean',
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0
)
fig.update_layout(xaxis_tickangle=45, height=500)
fig.show(renderer="notebook")

### Sentiment by State

In [ ]:
df_states_sent = df_states.copy()
df_states_sent['vader_compound'] = df_sample.loc[df_states_sent.index, 'vader_compound']

state_sentiment = df_states_sent.groupby('state_code')['vader_compound'].mean().reset_index()
state_sentiment.columns = ['state_code', 'mean_sentiment']

fig = px.choropleth(
    state_sentiment,
    locations='state_code',
    locationmode='USA-states',
    color='mean_sentiment',
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    scope='usa',
    title='Average Tweet Sentiment by State'
)
fig.update_layout(height=500)
fig.show(renderer="notebook")

### State x Topic Sentiment Heatmap

In [ ]:
df_cross = df_pol_states.copy()
df_cross['vader_compound'] = df_sample.loc[df_cross.index, 'vader_compound']

pivot = df_cross.groupby(['state_code', 'topic_label'])['vader_compound'].mean().unstack(fill_value=0)

fig = px.imshow(
    pivot,
    labels=dict(x="Political Topic", y="State", color="Sentiment"),
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    title="Sentiment Heatmap: State × Political Topic",
    aspect='auto'
)
fig.update_layout(height=800)
fig.show(renderer="notebook")

## 4. Zero-Shot Partisan Stance Classification

We use a pre-trained transformer model for zero-shot classification to detect partisan stance in political tweets. This captures information that neither BERTopic (what people discuss) nor VADER (emotional tone) can provide: **which political side** the tweet supports.

Due to computational constraints, we apply this to a subsample of political tweets and aggregate results at the state level.

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=DEVICE
)

candidate_labels = [
    "supports Republican party",
    "supports Democratic party",
    "neutral or nonpartisan"
]

print(f"Model loaded: facebook/bart-large-mnli (device={DEVICE})")
print(f"Candidate labels: {candidate_labels}")

In [ ]:
llm_sample_size = min(3000, len(df_political))
df_llm = df_political.sample(n=llm_sample_size, random_state=RANDOM_STATE).copy()

print(f"Classifying {len(df_llm)} political tweets...")
print("This may take 30-60 minutes on CPU.")

stance_results = []
batch_size = 8
texts = df_llm['tweet_clean'].tolist()

for i in tqdm(range(0, len(texts), batch_size), desc="Stance classification"):
    batch = texts[i:i+batch_size]
    results = classifier(batch, candidate_labels=candidate_labels, batch_size=batch_size)
    if not isinstance(results, list):
        results = [results]
    for r in results:
        stance_results.append({
            'stance_label': r['labels'][0],
            'stance_score': r['scores'][0]
        })

df_llm['stance_label'] = [r['stance_label'] for r in stance_results]
df_llm['stance_score'] = [r['stance_score'] for r in stance_results]

print(f"\nStance distribution:")
print(df_llm['stance_label'].value_counts())
print(f"\nMean confidence: {df_llm['stance_score'].mean():.3f}")

In [ ]:
print("High-confidence examples per stance:\n")
for label in candidate_labels:
    subset = df_llm[df_llm['stance_label'] == label].nlargest(3, 'stance_score')
    print(f"--- {label.upper()} ---")
    for _, row in subset.iterrows():
        print(f"  [{row['stance_score']:.2f}] {row['tweet_clean'][:120]}")
    print()

### State-Level Partisan Lean from LLM

In [ ]:
df_llm['state_code'] = df_sample.loc[df_llm.index, 'state_code']
df_llm_states = df_llm.dropna(subset=['state_code'])

state_stance = df_llm_states.groupby('state_code')['stance_label'].value_counts().unstack(fill_value=0)

for col in candidate_labels:
    if col not in state_stance.columns:
        state_stance[col] = 0

state_stance['total'] = state_stance.sum(axis=1)
state_stance['pct_dem'] = state_stance['supports Democratic party'] / state_stance['total']
state_stance['pct_rep'] = state_stance['supports Republican party'] / state_stance['total']
state_stance['partisan_lean'] = state_stance['pct_dem'] - state_stance['pct_rep']

min_tweets_for_lean = 10
state_stance_valid = state_stance[state_stance['total'] >= min_tweets_for_lean].copy()

fig = px.choropleth(
    state_stance_valid.reset_index(),
    locations='state_code',
    locationmode='USA-states',
    color='partisan_lean',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    scope='usa',
    title='LLM-Derived Partisan Lean by State (Blue=Dem, Red=Rep)',
    labels={'partisan_lean': 'Partisan Lean'}
)
fig.update_layout(height=500)
fig.show(renderer="notebook")

print(f"States with sufficient data: {len(state_stance_valid)}")
print(f"\nTop 5 Democratic-leaning states:")
print(state_stance_valid.nlargest(5, 'partisan_lean')[['partisan_lean', 'total']])
print(f"\nTop 5 Republican-leaning states:")
print(state_stance_valid.nsmallest(5, 'partisan_lean')[['partisan_lean', 'total']])

## 5. Feature Engineering and Election Data

We construct a state-level feature matrix combining outputs from all three methods (BERTopic, VADER, Zero-Shot Stance) with engagement metrics. The target variable is the 2024 presidential election outcome by state.

In [ ]:
election_2024 = {
    'AL': ('R', 25.2), 'AK': ('R', 14.9), 'AZ': ('R', 5.5), 'AR': ('R', 28.3),
    'CA': ('D', -19.5), 'CO': ('D', -11.4), 'CT': ('D', -12.4), 'DE': ('D', -11.0),
    'DC': ('D', -75.0), 'FL': ('R', 13.2), 'GA': ('R', 2.2), 'HI': ('D', -19.1),
    'ID': ('R', 32.3), 'IL': ('D', -7.5), 'IN': ('R', 20.0), 'IA': ('R', 13.5),
    'KS': ('R', 14.5), 'KY': ('R', 25.6), 'LA': ('R', 19.7), 'ME': ('D', -6.3),
    'MD': ('D', -21.0), 'MA': ('D', -23.2), 'MI': ('R', 1.4), 'MN': ('D', -2.3),
    'MS': ('R', 18.6), 'MO': ('R', 17.0), 'MT': ('R', 18.5), 'NE': ('R', 17.5),
    'NV': ('R', 3.2), 'NH': ('D', -1.2), 'NJ': ('D', -5.7), 'NM': ('D', -3.0),
    'NY': ('D', -12.2), 'NC': ('R', 3.3), 'ND': ('R', 27.8), 'OH': ('R', 11.1),
    'OK': ('R', 33.7), 'OR': ('D', -6.1), 'PA': ('R', 1.7), 'RI': ('D', -10.0),
    'SC': ('R', 14.5), 'SD': ('R', 22.0), 'TN': ('R', 25.8), 'TX': ('R', 13.6),
    'UT': ('R', 18.8), 'VT': ('D', -22.8), 'VA': ('D', -3.7), 'WA': ('D', -12.2),
    'WV': ('R', 32.0), 'WI': ('R', 0.9), 'WY': ('R', 43.0)
}

df_election = pd.DataFrame([
    {'state_code': k, 'winner': v[0], 'margin_r': v[1]}
    for k, v in election_2024.items()
])
df_election['outcome_binary'] = (df_election['winner'] == 'R').astype(int)

print(f"States: {len(df_election)}")
print(f"Republican wins: {(df_election['winner'] == 'R').sum()}")
print(f"Democratic wins: {(df_election['winner'] == 'D').sum()}")

### Build State-Level Feature Matrix

In [ ]:
features = {}

# 1. Topic distribution per state (% of political tweets in each topic)
for state in df_states['state_code'].unique():
    state_data = df_states[df_states['state_code'] == state]
    state_pol = state_data[state_data['topic'].isin(political_topic_ids)]
    total = len(state_pol) if len(state_pol) > 0 else 1

    feat = {'state_code': state, 'tweet_count': len(state_data), 'political_tweet_count': len(state_pol)}

    for tid in political_topic_ids:
        label = topic_labels[tid].replace(" ", "_").replace("/", "_").lower()
        feat[f'topic_pct_{label}'] = (state_pol['topic'] == tid).sum() / total * 100

    # 2. VADER sentiment
    state_sent = df_sample.loc[state_data.index, 'vader_compound']
    feat['sentiment_mean'] = state_sent.mean() if len(state_sent) > 0 else 0
    feat['sentiment_std'] = state_sent.std() if len(state_sent) > 1 else 0

    # 3. Engagement metrics
    for metric in ['public_metrics.like_count', 'public_metrics.retweet_count.tweets_historical',
                    'public_metrics.impression_count.tweets_historical']:
        if metric in state_data.columns:
            col_clean = metric.split('.')[-1].replace('.', '_')
            vals = pd.to_numeric(state_data[metric], errors='coerce')
            feat[f'eng_{col_clean}'] = vals.mean() if not vals.isna().all() else 0

    features[state] = feat

df_features = pd.DataFrame(features.values())

# 4. Add LLM partisan lean
lean_data = state_stance_valid[['partisan_lean']].reset_index()
df_features = df_features.merge(lean_data, on='state_code', how='left')
df_features['partisan_lean'] = df_features['partisan_lean'].fillna(0)

# 5. Merge with election data
df_model = df_features.merge(df_election, on='state_code', how='inner')

# Remove states with very few tweets
min_tweets = 30
df_model = df_model[df_model['tweet_count'] >= min_tweets]
print(f"States in model (>={min_tweets} tweets): {len(df_model)}")
print(f"  Republican: {(df_model['outcome_binary'] == 1).sum()}")
print(f"  Democrat: {(df_model['outcome_binary'] == 0).sum()}")

excluded = set(df_election['state_code']) - set(df_model['state_code'])
if excluded:
    print(f"  Excluded (low tweet count): {excluded}")

In [ ]:
feature_cols = [c for c in df_model.columns if c not in
    ['state_code', 'winner', 'margin_r', 'outcome_binary', 'tweet_count', 'political_tweet_count']]

corr = df_model[feature_cols + ['outcome_binary']].corr()['outcome_binary'].drop('outcome_binary').sort_values()

fig = px.bar(
    x=corr.values, y=corr.index,
    orientation='h',
    title="Feature Correlation with Republican Win (outcome_binary)",
    labels={"x": "Pearson Correlation", "y": "Feature"},
    color=corr.values,
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0
)
fig.update_layout(height=max(400, len(corr) * 25), yaxis=dict(tickfont=dict(size=9)))
fig.show(renderer="notebook")

## 6. Predictive Modeling

We predict state-level partisan outcomes (Republican vs. Democrat) using the feature matrix constructed from BERTopic topic distributions, VADER sentiment, LLM partisan lean, and engagement metrics.

Due to the small sample size (~45-50 states), we use Leave-One-Out Cross-Validation (LOOCV) for robust evaluation.

In [ ]:
X = df_model[feature_cols].fillna(0)
y = df_model['outcome_binary']

X = X.replace([np.inf, -np.inf], 0)

loo = LeaveOneOut()

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE))
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=100, max_depth=3, random_state=RANDOM_STATE))
    ]),
    'Gradient Boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(n_estimators=50, max_depth=2, learning_rate=0.1, random_state=RANDOM_STATE))
    ])
}

results = {}
for name, model in models.items():
    y_pred = cross_val_predict(model, X, y, cv=loo)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average='weighted')
    results[name] = {'accuracy': acc, 'f1_weighted': f1, 'predictions': y_pred}
    print(f"\n{name}:")
    print(f"  Accuracy: {acc:.3f}")
    print(f"  F1 (weighted): {f1:.3f}")
    print(f"  Confusion Matrix:")
    print(f"  {confusion_matrix(y, y_pred)}")

In [ ]:
print("=" * 60)
print("MODEL COMPARISON (LOOCV)")
print("=" * 60)
print(f"{'Model':<25} {'Accuracy':>10} {'F1 (wtd)':>10}")
print("-" * 60)
for name, res in results.items():
    print(f"{name:<25} {res['accuracy']:>10.3f} {res['f1_weighted']:>10.3f}")
print("=" * 60)

best_name = max(results, key=lambda k: results[k]['accuracy'])
print(f"\nBest model: {best_name} (Accuracy: {results[best_name]['accuracy']:.3f})")

### Feature Importance

In [ ]:
best_model = models[best_name]
best_model.fit(X, y)

if hasattr(best_model.named_steps['clf'], 'feature_importances_'):
    importances = best_model.named_steps['clf'].feature_importances_
elif hasattr(best_model.named_steps['clf'], 'coef_'):
    importances = np.abs(best_model.named_steps['clf'].coef_[0])
else:
    importances = np.zeros(len(feature_cols))

feat_imp = pd.DataFrame({'feature': feature_cols, 'importance': importances})
feat_imp = feat_imp.sort_values('importance', ascending=True)

fig = px.bar(
    feat_imp.tail(15),
    x='importance', y='feature',
    orientation='h',
    title=f'Top 15 Feature Importances ({best_name})',
    labels={'importance': 'Importance', 'feature': 'Feature'}
)
fig.update_layout(height=500)
fig.show(renderer="notebook")

### Prediction Map

In [ ]:
best_preds = results[best_name]['predictions']
df_model['predicted'] = best_preds
df_model['correct'] = (df_model['outcome_binary'] == df_model['predicted']).astype(int)

df_model['pred_label'] = df_model['predicted'].map({1: 'R', 0: 'D'})
df_model['result_label'] = df_model.apply(
    lambda r: f"Actual: {r['winner']}, Pred: {r['pred_label']}" +
              (" ✓" if r['correct'] else " ✗"), axis=1
)

df_model['color_val'] = df_model.apply(
    lambda r: 2 if r['correct'] and r['winner'] == 'R'
    else (0 if r['correct'] and r['winner'] == 'D'
    else 1), axis=1
)

fig = px.choropleth(
    df_model,
    locations='state_code',
    locationmode='USA-states',
    color='color_val',
    color_continuous_scale=['blue', 'yellow', 'red'],
    scope='usa',
    hover_data=['state_code', 'winner', 'pred_label', 'correct'],
    title=f'Predicted vs Actual 2024 Election Outcomes ({best_name})'
)
fig.update_layout(
    height=500,
    coloraxis_showscale=False
)
fig.show(renderer="notebook")

misclassified = df_model[df_model['correct'] == 0]
print(f"Misclassified states ({len(misclassified)}):")
for _, row in misclassified.iterrows():
    print(f"  {row['state_code']}: Actual={row['winner']}, Predicted={row['pred_label']}")

# 7. Existing audit and model evaluation

We audit our pipeline for four types of bias:
1. **Geographic bias** — Are some states over/under-represented?
2. **Engagement bias** — Do high-follower accounts distort the signal?
3. **Temporal bias** — Is the sample skewed toward specific time periods?
4. **Model fairness** — Does prediction accuracy vary across state groups?

### 7.1 Geographic Bias

In [ ]:
state_population_2020 = {
    'CA': 39538, 'TX': 29146, 'FL': 21538, 'NY': 20202, 'PA': 13002,
    'IL': 12812, 'OH': 11800, 'GA': 10712, 'NC': 10439, 'MI': 10078,
    'NJ': 9289, 'VA': 8632, 'WA': 7615, 'AZ': 7151, 'MA': 7030,
    'TN': 6910, 'IN': 6786, 'MD': 6178, 'MO': 6155, 'WI': 5894,
    'CO': 5774, 'MN': 5707, 'SC': 5119, 'AL': 5024, 'LA': 4658,
    'KY': 4506, 'OR': 4238, 'OK': 3960, 'CT': 3606, 'UT': 3272,
    'IA': 3190, 'NV': 3105, 'AR': 3012, 'MS': 2962, 'KS': 2937,
    'NM': 2118, 'NE': 1962, 'ID': 1901, 'WV': 1794, 'HI': 1456,
    'NH': 1378, 'ME': 1362, 'MT': 1085, 'RI': 1098, 'DE': 990,
    'SD': 887, 'ND': 779, 'AK': 733, 'DC': 690, 'VT': 643, 'WY': 577
}

total_pop = sum(state_population_2020.values())

tweet_counts = df_states.groupby('state_code').size().reset_index(name='tweets')
total_tweets = tweet_counts['tweets'].sum()

tweet_counts['pop'] = tweet_counts['state_code'].map(state_population_2020)
tweet_counts = tweet_counts.dropna(subset=['pop'])
tweet_counts['pct_tweets'] = tweet_counts['tweets'] / total_tweets * 100
tweet_counts['pct_pop'] = tweet_counts['pop'] / total_pop * 100
tweet_counts['representation_ratio'] = tweet_counts['pct_tweets'] / tweet_counts['pct_pop']

tweet_counts = tweet_counts.sort_values('representation_ratio', ascending=False)

print("Geographic Representation Analysis")
print("=" * 60)
print(f"{'State':<6} {'Tweets':>7} {'%Tweets':>8} {'%Pop':>6} {'Ratio':>7}")
print("-" * 60)
for _, row in tweet_counts.iterrows():
    flag = " ⚠" if row['representation_ratio'] < 0.3 or row['representation_ratio'] > 3.0 else ""
    print(f"{row['state_code']:<6} {int(row['tweets']):>7} {row['pct_tweets']:>7.1f}% {row['pct_pop']:>5.1f}% {row['representation_ratio']:>6.2f}{flag}")

fig = px.choropleth(
    tweet_counts,
    locations='state_code',
    locationmode='USA-states',
    color='representation_ratio',
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=1.0,
    scope='usa',
    title='Geographic Representation Ratio (1.0 = proportional)',
    labels={'representation_ratio': 'Ratio'}
)
fig.update_layout(height=500)
fig.show(renderer="notebook")

### 7.2 Engagement Bias (Power Users)

In [ ]:
if 'public_metrics.followers_count' in df_sample.columns:
    followers = pd.to_numeric(df_sample['public_metrics.followers_count'], errors='coerce')
    threshold = followers.quantile(0.99)

    df_sample['is_power_user'] = followers >= threshold

    print(f"Power user threshold (top 1%): {threshold:.0f} followers")
    print(f"Power users: {df_sample['is_power_user'].sum()}")
    print(f"Regular users: {(~df_sample['is_power_user']).sum()}")

    power_topics = df_sample[df_sample['is_power_user'] & df_sample['topic'].isin(political_topic_ids)]
    regular_topics = df_sample[~df_sample['is_power_user'] & df_sample['topic'].isin(political_topic_ids)]

    if len(power_topics) > 0 and len(regular_topics) > 0:
        power_dist = power_topics['topic_label'].value_counts(normalize=True).sort_index()
        regular_dist = regular_topics['topic_label'].value_counts(normalize=True).sort_index()

        comparison = pd.DataFrame({'power_users': power_dist, 'regular_users': regular_dist}).fillna(0)
        comparison['difference'] = comparison['power_users'] - comparison['regular_users']

        print("\nTopic distribution comparison:")
        print(comparison.round(3))

        fig = go.Figure()
        fig.add_trace(go.Bar(name='Power Users (top 1%)', x=comparison.index, y=comparison['power_users']))
        fig.add_trace(go.Bar(name='Regular Users', x=comparison.index, y=comparison['regular_users']))
        fig.update_layout(
            barmode='group',
            title='Topic Distribution: Power Users vs Regular Users',
            xaxis_tickangle=45, height=500
        )
        fig.show(renderer="notebook")
else:
    print("Followers column not found — skipping engagement bias analysis.")
    df_sample['is_power_user'] = False

### 7.3 Temporal Bias

In [ ]:
time_col = None
for col in ['created_at', 'created_at.tweets', 'sampling_date']:
    if col in df_sample.columns:
        time_col = col
        break

if time_col:
    df_sample['datetime'] = pd.to_datetime(df_sample[time_col], errors='coerce')
    df_time = df_sample.dropna(subset=['datetime'])

    weekly = df_time.set_index('datetime').resample('W').size().reset_index(name='count')

    fig = px.line(
        weekly, x='datetime', y='count',
        title='Tweet Volume Over Time (Weekly)',
        labels={'datetime': 'Date', 'count': 'Tweets per Week'}
    )
    fig.update_layout(height=400)
    fig.show(renderer="notebook")

    print(f"Date range: {df_time['datetime'].min()} to {df_time['datetime'].max()}")
    print(f"Tweets with valid timestamp: {len(df_time)}")
else:
    print("No timestamp column found — skipping temporal bias analysis.")

### 7.4 Model Fairness Across State Groups

In [ ]:
swing_states = ['AZ', 'GA', 'MI', 'NV', 'NC', 'PA', 'WI']
df_model['is_swing'] = df_model['state_code'].isin(swing_states)

regions = {
    'Northeast': ['CT', 'DE', 'ME', 'MD', 'MA', 'NH', 'NJ', 'NY', 'PA', 'RI', 'VT', 'DC'],
    'South': ['AL', 'AR', 'FL', 'GA', 'KY', 'LA', 'MS', 'NC', 'SC', 'TN', 'TX', 'VA', 'WV'],
    'Midwest': ['IL', 'IN', 'IA', 'KS', 'MI', 'MN', 'MO', 'NE', 'ND', 'OH', 'SD', 'WI'],
    'West': ['AK', 'AZ', 'CA', 'CO', 'HI', 'ID', 'MT', 'NV', 'NM', 'OR', 'UT', 'WA', 'WY']
}
state_to_region = {s: r for r, states in regions.items() for s in states}
df_model['region'] = df_model['state_code'].map(state_to_region)

print("Model Fairness Analysis")
print("=" * 60)

# Swing vs Safe
swing = df_model[df_model['is_swing']]
safe = df_model[~df_model['is_swing']]
if len(swing) > 0:
    print(f"\nSwing states accuracy: {swing['correct'].mean():.3f} (n={len(swing)})")
if len(safe) > 0:
    print(f"Safe states accuracy:  {safe['correct'].mean():.3f} (n={len(safe)})")

# By region
print(f"\nAccuracy by region:")
for region in ['Northeast', 'South', 'Midwest', 'West']:
    subset = df_model[df_model['region'] == region]
    if len(subset) > 0:
        print(f"  {region:<12}: {subset['correct'].mean():.3f} (n={len(subset)})")

### 7.5 Audited Model: Impact of Bias Reduction

In [ ]:
# Re-run best model without power users
if df_sample['is_power_user'].any():
    df_no_power = df_sample[~df_sample['is_power_user']].copy()
    df_no_power_states = df_no_power[~df_no_power["state"].isin(exclude_states)].dropna(subset=["state_code"])

    features_audited = {}
    for state in df_no_power_states['state_code'].unique():
        state_data = df_no_power_states[df_no_power_states['state_code'] == state]
        state_pol = state_data[state_data['topic'].isin(political_topic_ids)]
        total = len(state_pol) if len(state_pol) > 0 else 1

        feat = {'state_code': state, 'tweet_count': len(state_data)}

        for tid in political_topic_ids:
            label = topic_labels[tid].replace(" ", "_").replace("/", "_").lower()
            feat[f'topic_pct_{label}'] = (state_pol['topic'] == tid).sum() / total * 100

        state_sent = df_no_power.loc[state_data.index, 'vader_compound'] if 'vader_compound' in df_no_power.columns else pd.Series([0])
        feat['sentiment_mean'] = state_sent.mean() if len(state_sent) > 0 else 0
        feat['sentiment_std'] = state_sent.std() if len(state_sent) > 1 else 0
        feat['partisan_lean'] = state_stance_valid.loc[state, 'partisan_lean'] if state in state_stance_valid.index else 0

        features_audited[state] = feat

    df_feat_audited = pd.DataFrame(features_audited.values())
    df_model_audited = df_feat_audited.merge(df_election, on='state_code', how='inner')
    df_model_audited = df_model_audited[df_model_audited['tweet_count'] >= min_tweets]

    audit_feature_cols = [c for c in df_model_audited.columns if c not in
        ['state_code', 'winner', 'margin_r', 'outcome_binary', 'tweet_count']]

    X_audit = df_model_audited[audit_feature_cols].fillna(0).replace([np.inf, -np.inf], 0)
    y_audit = df_model_audited['outcome_binary']

    if len(X_audit) > 5:
        best_audit = models[best_name]
        y_pred_audit = cross_val_predict(best_audit, X_audit, y_audit, cv=LeaveOneOut())
        acc_audit = accuracy_score(y_audit, y_pred_audit)

        print("=" * 60)
        print("BIAS AUDIT: MODEL COMPARISON")
        print("=" * 60)
        print(f"{'Metric':<30} {'Baseline':>10} {'Audited':>10}")
        print("-" * 60)
        print(f"{'Accuracy':<30} {results[best_name]['accuracy']:>10.3f} {acc_audit:>10.3f}")
        print(f"{'States used':<30} {len(df_model):>10} {len(df_model_audited):>10}")
        print(f"{'Power users removed':<30} {'No':>10} {'Yes':>10}")
        print("=" * 60)
    else:
        print("Not enough states after filtering for audited model comparison.")
else:
    print("No power users identified — audited model same as baseline.")

# 8: Systematic bias audit

This section elaborates on section 7, and shifts the audit one level deeper. Instead of asking wether the model performs well, it asks where bias may enter the pipeline, and how that bias can be measured. 

Hence, this section operationalizes bias as distortion rather than random error. A limitation simply reduces what we can claim, while a bias pushes the analysis in a predictable direction. Following our weekly goal 7 and weekly goal 8 reasoning, the main risk is that BERTopic, VADER and the state-level predictive model may reproduce the visibility logic of Twitter/X: issues that are emotionally charged, frequently posted, highly retweeted, or amplified by active users may appear more politically important than issues that are widely felt but less publicly posted.

The created audit here at 8, tests five linked risks:

1. Keyword and language exclusion bias: the political filter may overrepresent formal political vocabulary and exclude political talk expressed through other vocabularies or languages.
2. Active-user concentration bias: a small number of users may generate a large share of tweets, making BERTopic measure posting intensity rather than breadth of concern.
3. Platform amplification bias: topics with high retweets and emotionally extreme sentiment may look salient because they travel well on Twitter/X, not because they are the top priorities of voters.
4. Missingness and state-exclusion bias: missing or self-stated location variables may remove groups whose location is harder to parse, which matters because the prediction target is state-level.
5. Mitigation trade-off: a user-level tweet cap tests whether reducing the influence of highly active users changes topic salience and model performance.

This makes the audit reportable in two ways, since it first gives measurable diagnostics, and second theoretically, it links those diagnostics to the critical claim that platform visibility is not the same as democratic representativeness.

In [ ]:
#audit helper functions
from IPython.display import display, Markdown
import math

def find_first_col(df_, candidates): #finds the first column in df_ that matches or contains any of the candidate strings
    if df_ is None:
        return None
    
    cols = list(df_.columns) #get list of columns in df_
    lower_map = {c.lower(): c for c in cols} #map lowercase column names to original names for exact match
    
    #exact match first
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    
    #contains match second
    for cand in candidates:
        for col in cols:
            if cand.lower() in col.lower():
                return col
    
    return None


def gini_coefficient(values): #calculates the Gini coefficient for a list of values, ignoring NaNs, treating all 0 as undefined
    x = np.asarray(values, dtype=float)
    x = x[~np.isnan(x)]
    
    if len(x) == 0 or np.all(x == 0):
        return np.nan
    
    x = np.sort(x)
    n = len(x)
    return ((2 * np.arange(1, n + 1) - n - 1) @ x) / (n * x.sum())


def safe_zscore(series): #calculates z-score for a pandas Series, returning 0 for all values if std is 0 or NaN
    s = pd.Series(series).astype(float)
    if s.std(ddof=0) == 0 or np.isnan(s.std(ddof=0)):
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.mean()) / s.std(ddof=0)


#column detection
text_audit_col = "tweet_clean" if "tweet_clean" in df_sample.columns else tweets_col

user_col = find_first_col(
    df_sample,
    [
        "user_id_historical_code",
        "author_id",
        "user_id",
        "users_id",
        "username",
        "screen_name"
    ]
)

retweet_col = find_first_col(
    df_sample,
    [
        "public_metrics.retweet_count.tweets_historical",
        "retweet_count",
        "retweets"
    ]
)

like_col = find_first_col(
    df_sample,
    [
        "public_metrics.like_count",
        "like_count",
        "likes"
    ]
)

impression_col = find_first_col(
    df_sample,
    [
        "public_metrics.impression_count.tweets_historical",
        "impression_count",
        "impressions"
    ]
)

language_col = find_first_col(
    df_sample,
    [
        "lang",
        "language",
        "tweet_language"
    ]
)

timestamp_col = find_first_col(
    df_sample,
    [
        "created_at",
        "created_at.tweets",
        "sampling_date",
        "date",
        "timestamp"
    ]
)

print("Detected audit columns")
print("=" * 60)
print(f"Text column:       {text_audit_col}")
print(f"User column:       {user_col}")
print(f"Retweet column:    {retweet_col}")
print(f"Like column:       {like_col}")
print(f"Impression column: {impression_col}")
print(f"Language column:   {language_col}")
print(f"Timestamp column:  {timestamp_col}")

In [ ]:
#Keyword filter and language exclusion audit

audit_text = df_sample[text_audit_col].fillna("").astype(str)

unique_keywords = sorted(set([kw.strip().lower() for kw in political_keywords if str(kw).strip()]))

keyword_rows = []

for kw in unique_keywords:
    #match whole words and allow flexible whitespace for multi-word keywords.
    kw_regex = r"(?i)(?<!\w)" + re.escape(kw).replace(r"\ ", r"\s+") + r"(?!\w)"
    hits = audit_text.str.contains(kw_regex, regex=True, na=False)
    
    keyword_rows.append({
        "keyword": kw,
        "tweet_hits": int(hits.sum()),
        "share_of_sample_pct": hits.mean() * 100
    })

keyword_audit = (
    pd.DataFrame(keyword_rows)
    .sort_values("tweet_hits", ascending=False)
    .reset_index(drop=True)
)

total_keyword_hits = keyword_audit["tweet_hits"].sum()

keyword_concentration = []
for n in [10, 20, 50]:
    n_actual = min(n, len(keyword_audit))
    share = keyword_audit.head(n_actual)["tweet_hits"].sum() / total_keyword_hits * 100 if total_keyword_hits > 0 else np.nan
    keyword_concentration.append({
        "top_n_keywords": n_actual,
        "share_of_all_keyword_hits_pct": share
    })

keyword_concentration = pd.DataFrame(keyword_concentration)

print("Keyword filter concentration")
print("=" * 60)
display(keyword_concentration.round(2))

print("\nTop 25 keyword hits")
display(keyword_audit.head(25).round(3))

fig = px.bar(
    keyword_audit.head(25).sort_values("tweet_hits"),
    x="tweet_hits",
    y="keyword",
    orientation="h",
    title="Top 25 political keyword hits in df_sample",
    labels={
        "tweet_hits": "Number of tweets containing keyword",
        "keyword": "Keyword"
    }
)
fig.update_layout(height=650)
fig.show(renderer="notebook")


#language exclusion diagnostic
print("\nLanguage exclusion diagnostic")
print("=" * 60)

if language_col is not None:
    language_audit = (
        df_sample[language_col]
        .fillna("missing")
        .astype(str)
        .value_counts()
        .reset_index()
    )
    language_audit.columns = ["language", "n_tweets"]
    language_audit["pct"] = language_audit["n_tweets"] / language_audit["n_tweets"].sum() * 100
    
    display(language_audit.head(15).round(2))
    
    fig = px.bar(
        language_audit.head(15),
        x="language",
        y="pct",
        title="Language distribution after preprocessing",
        labels={"language": "Detected language", "pct": "% of df_sample"}
    )
    fig.update_layout(height=400)
    fig.show(renderer="notebook")
else:
    print(
        "No language column was found in df_sample. "
        "Report this as an audit limitation: language exclusion cannot be measured directly from the retained sample."
    )


#interpretation flag
top10_share = keyword_concentration.loc[keyword_concentration["top_n_keywords"] == min(10, len(keyword_audit)), "share_of_all_keyword_hits_pct"]
top10_share = float(top10_share.iloc[0]) if len(top10_share) else np.nan

if not np.isnan(top10_share):
    if top10_share >= 60:
        print(
            f"\nInterpretation: the top 10 keywords account for {top10_share:.1f}% of all keyword hits. "
            "This suggests a concentrated political filter: the sample may be dominated by a narrow vocabulary."
        )
    else:
        print(
            f"\nInterpretation: the top 10 keywords account for {top10_share:.1f}% of all keyword hits. "
            "This suggests the political filter is not dominated entirely by the top terms, although exclusion bias may still exist."
        )

In [ ]:

#active user concentration audit


if user_col is None:
    print(
        "No user identifier column was found. "
        "The active-user concentration audit cannot be computed directly. "
        "For the final report, state this as a limitation because user-level overrepresentation cannot be measured."
    )
    user_activity = None
    activity_gini = np.nan

else:
    user_activity = (
        df_sample[user_col]
        .fillna("missing_user")
        .astype(str)
        .value_counts()
        .rename_axis("user_id")
        .reset_index(name="tweet_count")
    )

    activity_gini = gini_coefficient(user_activity["tweet_count"])

    print("Active-user concentration audit")
    print("=" * 60)
    print(f"User column used: {user_col}")
    print(f"Number of unique users: {len(user_activity):,}")
    print(f"Mean tweets per user: {user_activity['tweet_count'].mean():.2f}")
    print(f"Median tweets per user: {user_activity['tweet_count'].median():.2f}")
    print(f"Maximum tweets by one user: {user_activity['tweet_count'].max():,}")
    print(f"Gini coefficient of tweet production: {activity_gini:.3f}")

    top_share_rows = []
    for p in [0.01, 0.05, 0.10]:
        n_top = max(1, math.ceil(len(user_activity) * p))
        share = user_activity.head(n_top)["tweet_count"].sum() / user_activity["tweet_count"].sum() * 100
        top_share_rows.append({
            "top_user_share": f"Top {int(p * 100)}%",
            "n_users": n_top,
            "share_of_tweets_pct": share
        })
    
    top_user_share = pd.DataFrame(top_share_rows)
    display(top_user_share.round(2))

    #mark top 1% most active users for later use
    n_top_1pct = max(1, math.ceil(len(user_activity) * 0.01))
    top_1pct_users = set(user_activity.head(n_top_1pct)["user_id"])
    df_sample["audit_top_1pct_active_user"] = (
        df_sample[user_col]
        .fillna("missing_user")
        .astype(str)
        .isin(top_1pct_users)
    )

    #lorenz curve
    counts_sorted = np.sort(user_activity["tweet_count"].values)
    cumulative_tweets = np.insert(np.cumsum(counts_sorted) / counts_sorted.sum(), 0, 0)
    cumulative_users = np.arange(0, len(cumulative_tweets)) / (len(cumulative_tweets) - 1)

    lorenz_df = pd.DataFrame({
        "cumulative_share_users": cumulative_users,
        "cumulative_share_tweets": cumulative_tweets
    })

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=lorenz_df["cumulative_share_users"],
        y=lorenz_df["cumulative_share_tweets"],
        mode="lines",
        name="Observed tweet concentration"
    ))
    fig.add_trace(go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        line=dict(dash="dash"),
        name="Perfect equality"
    ))
    fig.update_layout(
        title=f"Lorenz curve of tweet production by user, Gini = {activity_gini:.3f}",
        xaxis_title="Cumulative share of users",
        yaxis_title="Cumulative share of tweets",
        height=500
    )
    fig.show(renderer="notebook")

    if activity_gini >= 0.60:
        print(
            "\nInterpretation: tweet production is highly concentrated. "
            "BERTopic may therefore reflect the posting behavior of highly active users more than broad voter concern."
        )
    elif activity_gini >= 0.40:
        print(
            "\nInterpretation: tweet production is moderately concentrated. "
            "Active-user bias should be considered when interpreting issue salience."
        )
    else:
        print(
            "\nInterpretation: tweet production is relatively dispersed compared with the high-concentration scenario, "
            "although representativeness is still not guaranteed."
        )

In [ ]:
#topic engagement cross-reference


df_amp = df_sample[df_sample["topic"].isin(political_topic_ids)].copy()

if "topic_label" not in df_amp.columns:
    df_amp["topic_label"] = df_amp["topic"].map(topic_labels)

if "vader_compound" not in df_amp.columns:
    print("vader_compound was not found. Run the VADER section before this audit cell.")
else:
    if retweet_col is not None:
        df_amp["audit_retweets"] = pd.to_numeric(df_amp[retweet_col], errors="coerce").fillna(0)
    else:
        df_amp["audit_retweets"] = 0
        print("Retweet column not found. Retweet-based amplification will be set to 0.")

    df_amp["audit_abs_sentiment"] = df_amp["vader_compound"].abs()

    agg_dict = {
        "n_tweets": ("topic_label", "size"),
        "mean_retweets": ("audit_retweets", "mean"),
        "median_retweets": ("audit_retweets", "median"),
        "mean_vader": ("vader_compound", "mean"),
        "mean_abs_sentiment": ("audit_abs_sentiment", "mean")
    }

    if user_col is not None:
        agg_dict["n_users"] = (user_col, lambda x: x.astype(str).nunique())

    topic_amp = (
        df_amp
        .groupby("topic_label")
        .agg(**agg_dict)
        .reset_index()
        .sort_values("n_tweets", ascending=False)
    )

    topic_amp["salience_pct"] = topic_amp["n_tweets"] / topic_amp["n_tweets"].sum() * 100

    if "n_users" in topic_amp.columns:
        topic_amp["breadth_ratio"] = topic_amp["n_users"] / topic_amp["n_tweets"]
        breadth_component = safe_zscore(1 - topic_amp["breadth_ratio"])
    else:
        topic_amp["breadth_ratio"] = np.nan
        breadth_component = 0

    topic_amp["amplification_risk_score"] = (
        safe_zscore(topic_amp["salience_pct"]) +
        safe_zscore(np.log1p(topic_amp["mean_retweets"])) +
        safe_zscore(topic_amp["mean_abs_sentiment"]) +
        breadth_component
    )

    topic_amp = topic_amp.sort_values("amplification_risk_score", ascending=False)

    print("Topic amplification cross-reference")
    print("=" * 60)
    display(topic_amp.round(3))

    fig = px.scatter(
        topic_amp,
        x="salience_pct",
        y="mean_retweets",
        size="n_tweets",
        color="mean_abs_sentiment",
        hover_name="topic_label",
        hover_data=["n_tweets", "mean_vader", "breadth_ratio", "amplification_risk_score"],
        title="Topic salience vs. retweet amplification and emotional intensity",
        labels={
            "salience_pct": "% of political tweets",
            "mean_retweets": "Mean retweets",
            "mean_abs_sentiment": "Mean absolute VADER sentiment"
        }
    )
    fig.update_layout(height=550)
    fig.show(renderer="notebook")

    riskiest_topic = topic_amp.iloc[0]["topic_label"]
    print(
        f"\nInterpretation: '{riskiest_topic}' has the highest amplification risk score. "
        "If this topic is also dominant in the predictive features, the report should distinguish "
        "between voter-perceived importance and platform-amplified salience."
    )

In [ ]:
#missingness and pipeline exclusion audit


print("Missingness audit in retained sample")
print("=" * 60)

missingness_audit = pd.DataFrame({
    "column": df_sample.columns,
    "missing_pct": df_sample.isna().mean().values * 100,
    "missing_n": df_sample.isna().sum().values
}).sort_values("missing_pct", ascending=False)

fully_missing = missingness_audit[missingness_audit["missing_pct"] == 100]
high_missing = missingness_audit[(missingness_audit["missing_pct"] >= 50) & (missingness_audit["missing_pct"] < 100)]

print(f"Columns that are 100% missing in df_sample: {len(fully_missing)}")
display(fully_missing)

print("\nColumns with at least 50% missingness in df_sample")
display(high_missing.head(25).round(2))


#pipeline exclusion funnel
sample_n = len(df_sample)

valid_state_n = len(df_states) if "df_states" in globals() else np.nan

political_topic_n = (
    len(df_sample[df_sample["topic"].isin(political_topic_ids)])
    if "topic" in df_sample.columns and "political_topic_ids" in globals()
    else np.nan
)

model_state_tweet_n = (
    len(df_states[df_states["state_code"].isin(df_model["state_code"])])
    if "df_states" in globals() and "df_model" in globals()
    else np.nan
)

exclusion_funnel = pd.DataFrame([
    {
        "stage": "Sample after cleaning and sampling",
        "n_tweets": sample_n,
        "share_of_sample_pct": 100.0
    },
    {
        "stage": "Tweets with parseable U.S. state",
        "n_tweets": valid_state_n,
        "share_of_sample_pct": valid_state_n / sample_n * 100 if sample_n else np.nan
    },
    {
        "stage": "Tweets assigned to political BERTopic topics",
        "n_tweets": political_topic_n,
        "share_of_sample_pct": political_topic_n / sample_n * 100 if sample_n else np.nan
    },
    {
        "stage": "Tweets in states retained by prediction model",
        "n_tweets": model_state_tweet_n,
        "share_of_sample_pct": model_state_tweet_n / sample_n * 100 if sample_n else np.nan
    }
])

print("\nPipeline exclusion funnel")
display(exclusion_funnel.round(2))

fig = px.bar(
    exclusion_funnel,
    x="stage",
    y="share_of_sample_pct",
    text="share_of_sample_pct",
    title="Pipeline inclusion funnel: which tweets remain visible to the model?",
    labels={
        "stage": "Pipeline stage",
        "share_of_sample_pct": "% of original df_sample"
    }
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(height=500, xaxis_tickangle=25)
fig.show(renderer="notebook")


state_exclusion_rate = 100 - exclusion_funnel.loc[
    exclusion_funnel["stage"] == "Tweets with parseable U.S. state",
    "share_of_sample_pct"
].iloc[0]

print(
    f"\nInterpretation: {state_exclusion_rate:.1f}% of sampled tweets do not enter the state-level analysis "
    "because they lack a parseable U.S. state. Since the prediction target is state-level electoral outcome, "
    "this is not just missing data; it defines whose political expression can be connected to the model."
)

In [ ]:
#bias mitigation: user level tweet cap
USER_TWEET_CAP = 10

def build_state_feature_matrix_from_sample(sample_df, sample_name="sample"): #builds the state level feature matrix for modeling from a given sample dataframe, applying the same processing as the original model pipeline. Returns the new model dataframe, feature matrix X, and target y
    tmp = sample_df.copy()

    if "state_code" not in tmp.columns:
        tmp["state_code"] = tmp["state"].map(state_name_to_code)

    valid = tmp[~tmp["state"].isin(exclude_states)].dropna(subset=["state_code"]).copy()

    features_new = {}

    for state in valid["state_code"].unique():
        state_data = valid[valid["state_code"] == state]
        state_pol = state_data[state_data["topic"].isin(political_topic_ids)]
        total_pol = len(state_pol) if len(state_pol) > 0 else 1

        feat = {
            "state_code": state,
            "tweet_count": len(state_data),
            "political_tweet_count": len(state_pol)
        }

        #topic distribution
        for tid in political_topic_ids:
            label = topic_labels[tid].replace(" ", "_").replace("/", "_").lower()
            feat[f"topic_pct_{label}"] = (state_pol["topic"] == tid).sum() / total_pol * 100

        #VADER sentiment
        if "vader_compound" in state_data.columns:
            vals = pd.to_numeric(state_data["vader_compound"], errors="coerce")
            feat["sentiment_mean"] = vals.mean() if len(vals) > 0 else 0
            feat["sentiment_std"] = vals.std() if len(vals) > 1 else 0
        else:
            feat["sentiment_mean"] = 0
            feat["sentiment_std"] = 0

        #engagement metrics
        for metric in [
            "public_metrics.like_count",
            "public_metrics.retweet_count.tweets_historical",
            "public_metrics.impression_count.tweets_historical"
        ]:
            if metric in state_data.columns:
                col_clean = metric.split(".")[-1].replace(".", "_")
                vals = pd.to_numeric(state_data[metric], errors="coerce")
                feat[f"eng_{col_clean}"] = vals.mean() if not vals.isna().all() else 0

        features_new[state] = feat

    df_features_new = pd.DataFrame(features_new.values())

    if len(df_features_new) == 0:
        return None, None, None

    #add LLM partisan lean if available.
    if "state_stance_valid" in globals() and "partisan_lean" in state_stance_valid.columns:
        lean_data = state_stance_valid[["partisan_lean"]].reset_index()
        df_features_new = df_features_new.merge(lean_data, on="state_code", how="left")
        df_features_new["partisan_lean"] = df_features_new["partisan_lean"].fillna(0)
    else:
        df_features_new["partisan_lean"] = 0

    df_model_new = df_features_new.merge(df_election, on="state_code", how="inner")
    df_model_new = df_model_new[df_model_new["tweet_count"] >= min_tweets].copy()

    #align to original model feature columns
    for col in feature_cols:
        if col not in df_model_new.columns:
            df_model_new[col] = 0

    X_new = df_model_new[feature_cols].fillna(0).replace([np.inf, -np.inf], 0)
    y_new = df_model_new["outcome_binary"]

    return df_model_new, X_new, y_new


if user_col is None:
    print(
        "User-level tweet cap cannot be applied because no user identifier was detected. "
        "Report this as a limitation of the mitigation stage."
    )

else:
    df_for_cap = df_sample.copy()

    #preserve temporal ordering when possible. Otherwise, the existing random sample order is used
    if timestamp_col is not None and timestamp_col in df_for_cap.columns:
        df_for_cap["_audit_datetime"] = pd.to_datetime(df_for_cap[timestamp_col], errors="coerce")
        df_for_cap = df_for_cap.sort_values([user_col, "_audit_datetime"])
    else:
        df_for_cap = df_for_cap.sort_values([user_col])

    df_capped = (
        df_for_cap
        .groupby(user_col, group_keys=False)
        .head(USER_TWEET_CAP)
        .copy()
    )

    if "_audit_datetime" in df_capped.columns:
        df_capped = df_capped.drop(columns=["_audit_datetime"])

    print("User-level tweet-cap mitigation")
    print("=" * 60)
    print(f"Cap applied: maximum {USER_TWEET_CAP} tweets per user")
    print(f"Original sample size: {len(df_sample):,}")
    print(f"Capped sample size:   {len(df_capped):,}")
    print(f"Tweets removed:       {len(df_sample) - len(df_capped):,}")

    original_gini = gini_coefficient(df_sample[user_col].fillna("missing_user").astype(str).value_counts())
    capped_gini = gini_coefficient(df_capped[user_col].fillna("missing_user").astype(str).value_counts())

    print(f"Original user-activity Gini: {original_gini:.3f}")
    print(f"Capped user-activity Gini:   {capped_gini:.3f}")

    #compare topic distributions before and after capping.
    baseline_topic_dist = (
        df_sample[df_sample["topic"].isin(political_topic_ids)]["topic_label"]
        .value_counts(normalize=True)
        .rename("baseline_pct")
        * 100
    )

    capped_topic_dist = (
        df_capped[df_capped["topic"].isin(political_topic_ids)]["topic_label"]
        .value_counts(normalize=True)
        .rename("capped_pct")
        * 100
    )

    topic_cap_compare = (
        pd.concat([baseline_topic_dist, capped_topic_dist], axis=1)
        .fillna(0)
        .reset_index()
        .rename(columns={"index": "topic_label"})
    )

    topic_cap_compare["absolute_change_pct_points"] = (
        topic_cap_compare["capped_pct"] - topic_cap_compare["baseline_pct"]
    )

    print("\nTopic distribution change after user cap")
    display(topic_cap_compare.sort_values("absolute_change_pct_points", key=abs, ascending=False).round(2))

    topic_cap_long = topic_cap_compare.melt(
        id_vars="topic_label",
        value_vars=["baseline_pct", "capped_pct"],
        var_name="sample",
        value_name="percentage"
    )

    fig = px.bar(
        topic_cap_long,
        x="topic_label",
        y="percentage",
        color="sample",
        barmode="group",
        title="Political topic distribution before and after user-level tweet cap",
        labels={
            "topic_label": "Topic",
            "percentage": "% of political tweets",
            "sample": "Sample"
        }
    )
    fig.update_layout(height=550, xaxis_tickangle=35)
    fig.show(renderer="notebook")

    #rebuild model features after cap and evaluate with the same best model.
    df_model_capped, X_capped, y_capped = build_state_feature_matrix_from_sample(df_capped, "user_capped")

    if df_model_capped is None or len(df_model_capped) <= 5 or y_capped.nunique() < 2:
        print(
            "Not enough state-level data after user cap to rerun the predictive model. "
            "The mitigation can still be reported as a representational correction, but not as a full performance comparison."
        )

    else:
        capped_pred = cross_val_predict(models[best_name], X_capped, y_capped, cv=LeaveOneOut())
        capped_acc = accuracy_score(y_capped, capped_pred)
        capped_f1 = f1_score(y_capped, capped_pred, average="weighted")

        mitigation_summary = pd.DataFrame([
            {
                "model_version": "Baseline",
                "n_tweets": len(df_sample),
                "n_states": len(df_model),
                "user_activity_gini": original_gini,
                "accuracy": results[best_name]["accuracy"],
                "f1_weighted": results[best_name]["f1_weighted"]
            },
            {
                "model_version": f"Audited: max {USER_TWEET_CAP} tweets/user",
                "n_tweets": len(df_capped),
                "n_states": len(df_model_capped),
                "user_activity_gini": capped_gini,
                "accuracy": capped_acc,
                "f1_weighted": capped_f1
            }
        ])

        print("\nBias mitigation impact on predictive model")
        display(mitigation_summary.round(3))

        fig = px.bar(
            mitigation_summary.melt(
                id_vars="model_version",
                value_vars=["user_activity_gini", "accuracy", "f1_weighted"],
                var_name="metric",
                value_name="value"
            ),
            x="metric",
            y="value",
            color="model_version",
            barmode="group",
            title="Baseline vs. audited model after user-level tweet cap",
            labels={"metric": "Metric", "value": "Value", "model_version": "Model version"}
        )
        fig.update_layout(height=450)
        fig.show(renderer="notebook")

        print(
            "\nInterpretation: compare accuracy/F1 with the reduction in user-activity Gini. "
            "A socially responsible model does not automatically mean the highest possible accuracy; "
            "the relevant question is whether a small performance change buys a meaningful reduction in representational distortion."
        )

In [ ]:
#audit findings table for report


audit_findings_rows = []

#keyword filter
if "keyword_concentration" in globals() and len(keyword_concentration) > 0:
    top10_val = keyword_concentration.iloc[0]["share_of_all_keyword_hits_pct"]
    audit_findings_rows.append({
        "bias_tested": "Keyword-filter concentration",
        "diagnostic": "Share of keyword hits produced by the top 10 political keywords",
        "main_result": f"{top10_val:.1f}% of keyword hits came from the top 10 keywords",
        "interpretation": "High concentration means the political sample may reflect a narrow formal vocabulary rather than all political concern.",
        "mitigation_or_response": "Report concentration; broaden keyword list or compare against survey priorities."
    })

#user activity
if "activity_gini" in globals() and not np.isnan(activity_gini):
    audit_findings_rows.append({
        "bias_tested": "Active-user concentration",
        "diagnostic": "Gini coefficient of tweet production per user",
        "main_result": f"Gini = {activity_gini:.3f}",
        "interpretation": "High Gini means BERTopic may measure posting intensity rather than breadth of voter concern.",
        "mitigation_or_response": f"Apply user-level tweet cap, here tested with max {USER_TWEET_CAP} tweets/user."
    })

#topic amplification
if "topic_amp" in globals() and len(topic_amp) > 0:
    top_amp_topic = topic_amp.iloc[0]["topic_label"]
    audit_findings_rows.append({
        "bias_tested": "Platform amplification",
        "diagnostic": "Topic salience cross-referenced with retweets and absolute VADER sentiment",
        "main_result": f"Highest amplification-risk topic: {top_amp_topic}",
        "interpretation": "A dominant topic with high engagement and emotional intensity may be platform-amplified rather than voter-prioritized.",
        "mitigation_or_response": "Interpret salience as discourse salience, not direct voter priority; compare with survey results where available."
    })

#missingness and exclusion
if "exclusion_funnel" in globals() and len(exclusion_funnel) > 0:
    state_row = exclusion_funnel[exclusion_funnel["stage"] == "Tweets with parseable U.S. state"]
    if len(state_row) > 0:
        state_share = state_row["share_of_sample_pct"].iloc[0]
        audit_findings_rows.append({
            "bias_tested": "State-location exclusion",
            "diagnostic": "Share of df_sample retained after requiring parseable U.S. state",
            "main_result": f"{state_share:.1f}% retained for state-level analysis",
            "interpretation": "Tweets without parseable state are excluded from the model, shaping whose political expression can affect state-level predictions.",
            "mitigation_or_response": "Report exclusion rate; treat state-level findings as conditional on parseable self-reported location."
        })

#mitigation
if "mitigation_summary" in globals() and len(mitigation_summary) == 2:
    acc_change = mitigation_summary.loc[1, "accuracy"] - mitigation_summary.loc[0, "accuracy"]
    gini_change = mitigation_summary.loc[1, "user_activity_gini"] - mitigation_summary.loc[0, "user_activity_gini"]
    
    audit_findings_rows.append({
        "bias_tested": "Mitigation trade-off",
        "diagnostic": "Change in user-activity Gini and model accuracy after user-level cap",
        "main_result": f"Accuracy change = {acc_change:+.3f}; Gini change = {gini_change:+.3f}",
        "interpretation": "The audited model should be judged by both predictive performance and reduced representational distortion.",
        "mitigation_or_response": "Use the capped model if the reduction in concentration is substantively meaningful relative to the performance trade-off."
    })

audit_findings = pd.DataFrame(audit_findings_rows)

print("Final audit findings table")
display(audit_findings)

## 8. Summary

In [ ]:
print("=" * 70)
print("FINAL RESULTS SUMMARY")
print("=" * 70)
print()
print("TOPIC MODELING (BERTopic)")
print(f"  Topics discovered: {len(topics_words)}")
print(f"  Coherence (c_v):   {coherence_score:.4f} (baseline: 0.4609)")
print(f"  Topic diversity:   {diversity_score:.4f} (baseline: 0.8972)")
print(f"  Outlier rate:      {pct_after:.1f}% (baseline: 51.4%)")
print(f"  Political topics:  {len(political_topic_ids)}")
print()
print("SENTIMENT ANALYSIS (VADER)")
print(f"  Mean compound:     {df_sample['vader_compound'].mean():.4f}")
print(f"  Positive:          {(df_sample['vader_label']=='positive').sum()}")
print(f"  Neutral:           {(df_sample['vader_label']=='neutral').sum()}")
print(f"  Negative:          {(df_sample['vader_label']=='negative').sum()}")
print()
print("STANCE DETECTION (Zero-Shot LLM)")
print(f"  Tweets classified: {len(df_llm)}")
print(f"  Mean confidence:   {df_llm['stance_score'].mean():.3f}")
for label in candidate_labels:
    count = (df_llm['stance_label'] == label).sum()
    print(f"  {label}: {count}")
print()
print("PREDICTIVE MODELING")
print(f"  States in model:   {len(df_model)}")
print(f"  Best model:        {best_name}")
print(f"  Accuracy (LOOCV):  {results[best_name]['accuracy']:.3f}")
print(f"  F1 (weighted):     {results[best_name]['f1_weighted']:.3f}")
print(f"  Misclassified:     {len(misclassified)} states")
print("=" * 70)

FINAL RESULTS SUMMARY

TOPIC MODELING (BERTopic)


NameError: name 'topics_words' is not defined

### Limitations

1. **Sample size**: Only ~50 states as prediction units — inherently limits model complexity and generalizability.
2. **Twitter/X representativeness**: The platform's user base skews younger, more urban, and more politically engaged than the general electorate.
3. **Geographic coverage**: Small states may have insufficient tweet volume for reliable feature estimation.
4. **Temporal scope**: Tweets from a single election cycle; patterns may not generalize across cycles.
5. **Zero-shot classification**: Pre-trained model may carry biases from its training data. No domain-specific fine-tuning was performed.
6. **Keyword filtering**: Political keyword pre-filtering may miss subtle political content or capture false positives.
7. **State attribution**: State labels derived from user profiles may not reflect where users actually vote.

## 9. Export Artifacts for Streamlit App

Save all trained models and pre-computed data, then upload to HuggingFace so the Streamlit app can load them.

**Two options:**
- **Option A**: Upload to HuggingFace Hub (for HF Spaces deployment)
- **Option B**: Save locally / to Google Drive

In [ ]:
import joblib
import os

EXPORT_DIR = "app_artifacts"
os.makedirs(EXPORT_DIR, exist_ok=True)

# 1. Save BERTopic model
topic_model.save(os.path.join(EXPORT_DIR, "topic_model"))
print("Saved: topic_model/")

# 2. Save best predictive model
best_model.fit(X, y)
joblib.dump(best_model, os.path.join(EXPORT_DIR, "best_model.pkl"))
print(f"Saved: best_model.pkl ({best_name})")

# 3. Save DataFrames
export_cols_sample = ['tweet_clean', 'topic', 'topic_label', 'vader_compound',
                      'vader_label', 'state_code', 'is_political']
if 'is_power_user' in df_sample.columns:
    export_cols_sample.append('is_power_user')
available_cols = [c for c in export_cols_sample if c in df_sample.columns]
df_sample[available_cols].to_parquet(os.path.join(EXPORT_DIR, "df_sample.parquet"), index=False)
print(f"Saved: df_sample.parquet ({len(df_sample)} rows)")

df_model.to_parquet(os.path.join(EXPORT_DIR, "df_model.parquet"), index=False)
print(f"Saved: df_model.parquet ({len(df_model)} rows)")

feat_imp.to_parquet(os.path.join(EXPORT_DIR, "feat_imp.parquet"), index=False)
print("Saved: feat_imp.parquet")

# 4. Save metadata as JSON
with open(os.path.join(EXPORT_DIR, "topic_labels.json"), "w") as f:
    json.dump({str(k): v for k, v in topic_labels.items()}, f, indent=2)

with open(os.path.join(EXPORT_DIR, "political_topic_ids.json"), "w") as f:
    json.dump(political_topic_ids, f)

model_metrics = {
    "coherence": round(coherence_score, 4),
    "diversity": round(diversity_score, 4),
    "outlier_pct": round(pct_after, 1),
    "num_topics": len(topics_words),
    "best_model": best_name,
    "best_accuracy": round(results[best_name]['accuracy'], 3),
    "best_f1": round(results[best_name]['f1_weighted'], 3),
    "model_results": {
        name: {"accuracy": round(r['accuracy'], 3), "f1_weighted": round(r['f1_weighted'], 3)}
        for name, r in results.items()
    }
}
with open(os.path.join(EXPORT_DIR, "model_metrics.json"), "w") as f:
    json.dump(model_metrics, f, indent=2)

state_meta = {
    "swing_states": ["AZ", "GA", "MI", "NV", "NC", "PA", "WI"],
    "regions": {
        "Northeast": ["CT", "DE", "ME", "MD", "MA", "NH", "NJ", "NY", "PA", "RI", "VT", "DC"],
        "South": ["AL", "AR", "FL", "GA", "KY", "LA", "MS", "NC", "SC", "TN", "TX", "VA", "WV"],
        "Midwest": ["IL", "IN", "IA", "KS", "MI", "MN", "MO", "NE", "ND", "OH", "SD", "WI"],
        "West": ["AK", "AZ", "CA", "CO", "HI", "ID", "MT", "NV", "NM", "OR", "UT", "WA", "WY"]
    },
    "election_2024": {k: str(v) for k, v in election_2024.items()}
}
with open(os.path.join(EXPORT_DIR, "state_metadata.json"), "w") as f:
    json.dump(state_meta, f, indent=2)

print("\nAll JSON metadata saved.")
print(f"\nArtifacts directory contents:")
for root, dirs, files in os.walk(EXPORT_DIR):
    for fname in files:
        fpath = os.path.join(root, fname)
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f"  {os.path.relpath(fpath, EXPORT_DIR)}: {size_mb:.1f} MB")

### Option A: Upload artifacts to HuggingFace Hub

This uploads the artifacts to a HuggingFace dataset repo so the Streamlit Space can download them at startup. Run the cell below and follow the login prompt.

In [ ]:
from huggingface_hub import HfApi, login

# Login to HuggingFace (will prompt for token)
# Get your token from: https://huggingface.co/settings/tokens
login()

api = HfApi()

# Create or use existing repo for artifacts
REPO_ID = "Diogo2110/issue-salience-artifacts"  # Change if needed
REPO_TYPE = "dataset"

try:
    api.create_repo(repo_id=REPO_ID, repo_type=REPO_TYPE, exist_ok=True)
    print(f"Repo ready: {REPO_ID}")
except Exception as e:
    print(f"Repo creation note: {e}")

# Upload all artifacts
api.upload_folder(
    folder_path=EXPORT_DIR,
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
)

print(f"\nAll artifacts uploaded to: https://huggingface.co/datasets/{REPO_ID}")
print("The Streamlit app will download these at startup.")

### Option B: Save to Google Drive (Colab only)

If you prefer to download the artifacts manually instead of using HuggingFace Hub.

In [ ]:
# Mount Google Drive and copy artifacts
try:
    from google.colab import drive
    drive.mount('/content/drive')

    import shutil
    drive_dest = '/content/drive/MyDrive/issue_salience_artifacts'
    if os.path.exists(drive_dest):
        shutil.rmtree(drive_dest)
    shutil.copytree(EXPORT_DIR, drive_dest)
    print(f"Artifacts saved to Google Drive: {drive_dest}")
except ImportError:
    print("Not running on Colab. Artifacts are saved locally in:", EXPORT_DIR)
    print("You can zip and download them:")
    print(f"  !zip -r artifacts.zip {EXPORT_DIR}")